Let's break down the types of LLM metrics, their pros and cons, and how to interpret them.  LLM evaluation is a complex and evolving field, so this is a general overview.

# I. Categories of LLM Metrics

LLM metrics can be broadly categorized into:

Text Quality Metrics: These assess the quality of the generated text itself.

Task-Specific Metrics: These evaluate how well the LLM performs on a particular task.

Efficiency Metrics: These measure the computational resources required by the LLM.

Safety and Bias Metrics: These assess the potential for harmful or biased outputs.

# II. Text Quality Metrics

## Perplexity
Measures how well a language model predicts the next word in a sequence.  Lower perplexity is generally better.

Advantages: Easy to calculate, widely used.
Disadvantages: Doesn't directly correlate with human judgment of quality, sensitive to vocabulary and text domain.
Interpretation: Perplexity is exp(average cross-entropy loss). A perplexity of 20 means the model is, on average, as uncertain as if it were choosing uniformly among 20 tokens at each step (a perfect model scores 1). Comparing perplexity scores between models trained on the same data can be useful.
## BLEU (Bilingual Evaluation Understudy)
Measures the overlap of n-grams between the generated text and one or more reference texts.

Advantages: Easy to calculate, widely used.
Disadvantages: Doesn't capture semantic similarity, sensitive to word order variations, can be less reliable for highly creative text.
Interpretation: BLEU scores range from 0 to 1 (or 0 to 100). Higher scores indicate better overlap with the reference text. BLEU is the geometric mean of 1- to 4-gram precisions multiplied by a brevity penalty for short outputs, so a score of 0.5 does not simply mean "50% overlap"; scores are only comparable on the same references and tokenization.
## ROUGE (Recall-Oriented Understudy for Gisting Evaluation)
Similar to BLEU, but focuses on recall (how much of the reference text is captured by the generated text).  Variations include ROUGE-L (longest common subsequence), ROUGE-1 (unigram overlap), ROUGE-2 (bigram overlap).

Advantages: Considers different aspects of overlap than BLEU, various ROUGE scores offer different perspectives.
Disadvantages: Still relies on n-gram matching, doesn't capture semantic similarity.
Interpretation: ROUGE scores also range from 0 to 1. Higher scores are better. ROUGE-L measures how well the generated text captures the longest common subsequence of the reference text.
## METEOR (Metric for Evaluation of Translation)
Considers synonyms and paraphrases, attempting to capture semantic similarity.

Advantages: Better at capturing meaning than BLEU and ROUGE.
Disadvantages: More complex to calculate.
Interpretation: METEOR scores range from 0 to 1. Higher is better.
## BERTScore
Uses contextualized word embeddings from BERT to measure similarity between generated and reference texts at a semantic level.

Advantages: Captures semantic similarity well.
Disadvantages: Computationally more expensive.
Interpretation: BERTScore outputs a similarity score between 0 and 1. Higher is better.
## LLM-as-a-Judge
A strong LLM is prompted with a rubric to score or compare outputs (see `12LLMReview.py`).

Advantages: Scales far better than human review, can judge open-ended qualities (helpfulness, faithfulness).
Disadvantages: Judges have biases (position, verbosity, favoring their own style); validate against a sample of human ratings.
## Human Evaluation
The gold standard.  Humans judge the quality of the generated text based on criteria like fluency, coherence, relevance, and creativity.

Advantages: Most accurate measure of quality.
Disadvantages: Expensive, time-consuming, subjective, and difficult to scale.
Interpretation: Human evaluation often involves assigning scores on Likert scales (e.g., from 1 to 5) or ranking different outputs.
# III. Task-Specific Metrics

These metrics depend on the specific task the LLM is designed for.  Examples:

Accuracy, Precision, Recall, F1-score: Common metrics for classification tasks.
Exact Match: Used for question answering.
Mean Average Precision (MAP): Used for information retrieval.
Reward Models: Used in Reinforcement Learning from Human Feedback (RLHF) to align LLM outputs with human preferences.
# IV. Efficiency Metrics

Inference Speed: How quickly the LLM generates text (time-to-first-token, tokens per second).
Memory Usage: The amount of memory required by the LLM.
Computational Cost: The amount of computing power needed to run the LLM.
# V. Safety and Bias Metrics

Toxicity: Measures the presence of toxic or offensive language in the generated text.
Bias: Assesses whether the LLM exhibits biases towards certain groups of people. This is a complex and nuanced area. There are many types of bias.
Adversarial Attacks: Measures the robustness of the LLM to malicious inputs designed to elicit harmful or biased outputs.
# How to Interpret Values

Comparison is Key: Most metrics are most useful for comparing different LLMs or different versions of the same LLM.
Context Matters: The interpretation of a metric depends on the specific task and data. A BLEU score of 0.5 might be good for one task but not for another.
No Single Perfect Metric: LLM evaluation is multi-faceted. It's important to consider multiple metrics to get a comprehensive view of performance.
Human Evaluation is Crucial: While automated metrics are useful, human evaluation is essential for assessing the overall quality and usefulness of LLM outputs.
Important Note: The field of LLM evaluation is constantly evolving. New metrics and techniques are being developed all the time.  It's important to stay up-to-date with the latest research in this area.

# Example: scoring candidate outputs
We give GPT-2 a prompt, let it **generate** a continuation, and compare it with hand-written candidates using:
perplexity, BLEU, ROUGE, embedding-based semantic similarity, a simple keyword precision, and generation speed.

Requires `pip install torch transformers nltk rouge_score`.

In [1]:
import math
import time
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pre-trained model and tokenizer
model_name = "gpt2"  # Or any other causal LM
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()

In [2]:
# 1. Perplexity

def calculate_perplexity(text, context=None):
    """Perplexity of `text`. If `context` is given, only the tokens of `text` are scored
    (the context is fed in but masked out of the loss), i.e. how surprising the text is *given* the context."""
    if context is None:
        input_ids = tokenizer(text, return_tensors='pt').input_ids
        labels = input_ids.clone()
    else:
        context_ids = tokenizer(context, return_tensors='pt').input_ids
        text_ids = tokenizer(" " + text, return_tensors='pt').input_ids
        input_ids = torch.cat([context_ids, text_ids], dim=1)
        labels = input_ids.clone()
        labels[:, :context_ids.shape[1]] = -100  # -100 = ignored by the loss
    with torch.no_grad():
        loss = model(input_ids.to(device), labels=labels.to(device)).loss
    return math.exp(loss.item())


# 2. BLEU Score

def calculate_bleu(reference_text, generated_text):
    """Calculates the BLEU score between a reference and generated text."""
    reference_tokens = reference_text.lower().split()  # Simple tokenization
    generated_tokens = generated_text.lower().split()
    # Smoothing function to handle cases with few or no matching n-grams
    smoothing = SmoothingFunction().method4
    return sentence_bleu([reference_tokens], generated_tokens, smoothing_function=smoothing)


# 3. ROUGE Scores (F-measure)

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def calculate_rouge(reference_text, generated_text):
    scores = scorer.score(reference_text, generated_text)
    return {name: s.fmeasure for name, s in scores.items()}


# 4. Semantic Similarity (embedding cosine similarity - the idea behind BERTScore)

embed_name = "sentence-transformers/all-MiniLM-L6-v2"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_name)
embed_model = AutoModel.from_pretrained(embed_name).to(device).eval()

def embed(text):
    enc = embed_tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
        hidden = embed_model(**enc).last_hidden_state
    mask = enc['attention_mask'].unsqueeze(-1)
    return (hidden * mask).sum(1) / mask.sum(1)  # mean pooling over tokens

def calculate_semantic_similarity(reference_text, generated_text):
    return torch.nn.functional.cosine_similarity(embed(reference_text), embed(generated_text)).item()


# 5. Context Precision (Simplified Example)

def calculate_context_precision(context, generated_text, relevant_keywords):
    """Fraction of generated words that are relevant keywords also present in the context."""
    words = lambda s: set(s.lower().replace('.', '').replace(',', '').split())  # Simplified keyword extraction
    generated_keywords = words(generated_text)
    if len(generated_keywords) == 0:  # Avoid division by zero
        return 0.0
    relevant_generated = generated_keywords & set(relevant_keywords) & words(context)
    return len(relevant_generated) / len(generated_keywords)

In [3]:
# Example Data (Replace with your actual data)
context = "The cat sat on the mat."
reference_text = "The cat sat on the mat and purred."
relevant_keywords = ["cat", "mat", "sat", "purred"]

# Let the model generate a continuation of the context and time it (an efficiency metric)
inputs = tokenizer(context, return_tensors='pt').to(device)
start = time.perf_counter()
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=20, do_sample=True, top_p=0.9,
                                pad_token_id=tokenizer.eos_token_id)
elapsed = time.perf_counter() - start
new_tokens = output_ids.shape[1] - inputs.input_ids.shape[1]
gpt2_text = context + tokenizer.decode(output_ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f"GPT-2 generated: {gpt2_text!r}")
print(f"Generation speed: {new_tokens / elapsed:.1f} tokens/sec on {device}")

GPT-2 generated: "The cat sat on the mat. The cat was a little girl in pink, so I'd say it was a little girl.\n"
Generation speed: 32.3 tokens/sec on cpu


In [4]:
candidates = {
    "exact copy of context": "The cat sat on the mat.",
    "small word change": "The cat sat on a mat.",
    "paraphrase": "A kitten was resting on the rug, purring happily.",
    "unrelated": "Quarterly revenue grew by twelve percent.",
    "gpt2 generated": gpt2_text,
}

rows = []
for name, generated_text in candidates.items():
    rows.append({
        "candidate": name,
        "perplexity": calculate_perplexity(generated_text),
        "BLEU": calculate_bleu(reference_text, generated_text),
        **calculate_rouge(reference_text, generated_text),
        "semantic_sim": calculate_semantic_similarity(reference_text, generated_text),
        "context_precision": calculate_context_precision(context, generated_text, relevant_keywords),
    })

results = pd.DataFrame(rows).set_index("candidate")
results.round(3)

,perplexity,BLEU,rouge1,rouge2,rougeL,semantic_sim,context_precision
candidate,,,,,,,
exact copy of context,90.243,0.544,0.857,0.833,0.857,0.915,0.6
small word change,139.687,0.364,0.714,0.500,0.714,0.908,0.5
paraphrase,87.795,0.063,0.353,0.133,0.353,0.650,0.0
unrelated,158.336,0.000,0.000,0.000,0.000,0.009,0.0
gpt2 generated,19.194,0.162,0.387,0.345,0.387,0.609,0.2


## Reading the table
* **BLEU / ROUGE** reward exact word overlap: one changed word ("the" -> "a") costs a lot, and the *paraphrase* gets a BLEU near zero even though it means nearly the same thing.
* **Semantic similarity** (embeddings) still rates the paraphrase far closer to the reference than the *unrelated* sentence.
* **Perplexity** measures how predictable text is to GPT-2, not whether it is correct or on-topic: GPT-2's own rambling output usually gets the *lowest* perplexity, because it is literally what the model expected.
* No single number tells the whole story - which is why evaluation suites combine several metrics plus human or LLM-judge review.

In [5]:
# Conditional perplexity: how expected is each continuation *given* the context?
for name, generated_text in [("reference continuation", "It purred."),
                             ("unrelated continuation", "Quarterly revenue grew by twelve percent.")]:
    print(f"{name:24s} perplexity given context = {calculate_perplexity(generated_text, context=context):.1f}")

reference continuation   perplexity given context = 33.7
unrelated continuation   perplexity given context = 303.3
